# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WingManOO7/ML-INTER/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

**Lane 2 — Refresh / Content Opportunity Scoring** (framed in ML-02 and ML-03).

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup — open the warehouse (the token never appears in a cell)

The token comes from the `HF_TOKEN` environment variable or a Colab Secret named `HF_TOKEN`.
I fetch only what this notebook needs: **one mid-panel month** of the daily fact table
(`month=2026-03`, about 124 MB) and `dim_content` (about 20 MB). They land in the Hugging Face
cache, so re-runs read local files and never re-scan the release (no HTTP 429).

June 2026 — the `_sample` table, the panel's last month — is my **sealed test month**. It is
never opened here: it is the natural outcome window of any past → future label.

In [1]:
import os, sys, subprocess, importlib.util, json
from pathlib import Path

if importlib.util.find_spec('duckdb') is None:      # Colab ships duckdb; this covers fresh local envs
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'duckdb'])

import duckdb
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download, list_repo_files

# Token order: env var -> Colab Secret -> prompt (last resort). Never paste a token into a cell.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
if not HF_TOKEN:
    import getpass
    HF_TOKEN = getpass.getpass('Hugging Face READ token: ')

REPO = 'FlyRank/internship-warehouse'
DEV_MONTH = '2026-03'          # mid-panel development month; 2026-06 is the sealed test month

def fetch(path):
    # downloads once into the HF cache; later runs reuse the local copy
    return Path(hf_hub_download(REPO, path, repo_type='dataset', token=HF_TOKEN)).as_posix()

month_files = [f for f in list_repo_files(REPO, repo_type='dataset', token=HF_TOKEN)
               if f.startswith(f'fact_content_daily_performance/month={DEV_MONTH}/')]

con = duckdb.connect()
con.execute('SET enable_progress_bar = false')
con.execute(f"CREATE OR REPLACE VIEW daily AS SELECT * FROM read_parquet({[fetch(f) for f in month_files]})")
con.execute(f"CREATE OR REPLACE VIEW dim_content AS SELECT * FROM read_parquet('{fetch('dim_content.parquet')}')")
sql = lambda q: con.sql(q).df()

print(f'daily  <- month={DEV_MONTH} ({len(month_files)} parquet file)')
print('dim_content <- dim_content.parquet')

daily  <- month=2026-03 (1 parquet file)
dim_content <- dim_content.parquet


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Contract answer 1 — what one row means.**
- In the raw table `fact_content_daily_performance`: one row = **one content item, of one client, on
  one `report_date`** (key `report_date + client_hash_id + content_hash_id`). Check 1/3 proves it.
- In my feature frame: one row = **one content item as seen at the decision moment T** — its two weeks
  of search data before T rolled up into five numbers, plus its label from the two weeks after T.
  That is the grain an editor acts on: pages, not page-days.

**Contract answer 2 — which tables.** `fact_content_daily_performance`, only the `month=2026-03`
partition, for the search metrics. `dim_content` for exactly one column, `content_created_date`
(page age). Nothing else is joined.

**Contract answer 3 — which time window.**

| Window | Dates | Days | Used for |
|---|---|---:|---|
| Feature window | Mon 2026-03-02 → Sun 2026-03-15 | 14 | the five features, nothing else |
| **Decision moment T** | **Mon 2026-03-16** | — | everything before T is "known" |
| Outcome window | Mon 2026-03-16 → Sun 2026-03-29 | 14 | the label, nothing else |
| Unused buffer | 03-01, 03-30, 03-31 | 3 | keeps both windows to whole Mon–Sun weeks |
| Sealed | June 2026 (the `_sample` month) | — | final test month, never opened while developing |

Both windows are exactly two Mon–Sun weeks, so a different weekday/weekend mix cannot fake a drop.
This is a one-month, scaled-down version of the capstone shape from ML-03 (90 days before T → 30 days
after T). The cell below checks the windows don't overlap.

In [2]:
T = pd.Timestamp('2026-03-16')
FEAT_START, FEAT_END = pd.Timestamp('2026-03-02'), pd.Timestamp('2026-03-15')
OUT_START,  OUT_END  = pd.Timestamp('2026-03-16'), pd.Timestamp('2026-03-29')

assert FEAT_END < T <= OUT_START, 'features must end before T; the label starts at T'

rows = []
for name, start, end in [('feature', FEAT_START, FEAT_END), ('outcome', OUT_START, OUT_END)]:
    days = pd.date_range(start, end)
    rows.append({'window': name, 'start': start.date(), 'end': end.date(), 'days': len(days),
                 'starts_on': start.day_name(), 'weekend_days': int((days.dayofweek >= 5).sum())})
pd.DataFrame(rows)

,window,start,end,days,starts_on,weekend_days
0,feature,2026-03-02,2026-03-15,14,Monday,4
1,outcome,2026-03-16,2026-03-29,14,Monday,4


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Contract answer 4 — what I predict / rank.** `declined_next_2w` = 1 when a page's GSC impressions in
the outcome window are **more than 20% below** its impressions in the feature window
(`imp_out < 0.8 × imp_pre`). I score only pages with **≥ 100 impressions** in the feature window
(the ML-03 volume floor; below it, a 20% swing is mostly noise) that were tracked from the window's
first day. Ranking pages by predicted risk gives the editor's review queue, judged by **precision@50**
(ML-03). The label is a measured proxy for "this page is losing search visibility". It does not show
that a refresh would bring the impressions back.

**Contract answer 5 — one thing I deliberately exclude: `dim_content.content_updated_date`** (and its
cousins `last_optimized_date`, `is_published`, `is_deleted`). `dim_content` is a snapshot taken at
export (2026-07-03). These columns describe each page *as of July*, not as of T. If a page was updated
in April, the column already says April. Feeding it to a model at T means reading the future. The
query under this cell measures how often that happens.

| Bucket | Field (table) | Window / note |
|---|---|---|
| Feature | `gsc_impressions` (daily) | feature window only |
| Feature | `gsc_sum_position`, `gsc_avg_position` (daily) | feature window only; position 0 = "no data", skipped |
| Feature | `content_created_date` (dim_content) | before T for every scored page (asserted in 3) |
| **Label** | `gsc_impressions` (daily) | **outcome window only** — same column, later dates, never a feature |
| Context | `client_hash_id`, `content_hash_id` | join, group, grouped split — pseudonyms, never features |
| Context | `report_date`, `gsc_data_available` | puts rows into a window; row filter with `IS TRUE` |
| Excluded | `content_updated_date`, `last_optimized_date`, `is_published`, `is_deleted` | July snapshot → future information at T |
| Excluded | `keyword_hash_id`, `url_hash_id` | IDs: grouping and dedup only |
| Excluded | `search_volume`, `cpc`, `competition`, `competition_level` | keyword-tool values with no date; I can't show they were known at T |
| Excluded | every GA4, `sessions_*`, `ai_*`, `scroll_events` column | outside this search question; GA4 availability is FALSE or NULL on most rows (Check 3/3) |
| Excluded | `gsc_clicks` | not needed for a visibility label; CTR belongs to Lane 4 |

In [3]:
schema = pd.concat([sql('DESCRIBE daily').assign(table='daily'),
                    sql('DESCRIBE dim_content').assign(table='dim_content')])[['table', 'column_name']]

touched = pd.DataFrame([
    ('daily', 'gsc_impressions', 'feature (pre-T) + label (post-T)'),
    ('daily', 'gsc_sum_position', 'feature'),
    ('daily', 'gsc_avg_position', 'feature'),
    ('dim_content', 'content_created_date', 'feature'),
    ('daily', 'client_hash_id', 'context'),
    ('daily', 'content_hash_id', 'context'),
    ('daily', 'report_date', 'context'),
    ('daily', 'gsc_data_available', 'context'),
    ('dim_content', 'content_updated_date', 'EXCLUDED'),
    ('dim_content', 'last_optimized_date', 'EXCLUDED'),
    ('dim_content', 'is_published', 'EXCLUDED'),
    ('dim_content', 'is_deleted', 'EXCLUDED'),
], columns=['table', 'column_name', 'bucket'])

missing = touched.merge(schema, how='left', indicator=True).query("_merge == 'left_only'")
assert missing.empty, f'bucketed a column that does not exist: {missing}'
print(f'{len(schema)} columns in the two tables; {len(touched)} touched or named — every one exists and has a bucket.')
print('Everything else stays out by default.\n')

# Evidence for answer 5: how often does the July snapshot's update date already lie AFTER T?
sql(f'''
    SELECT COUNT(*) AS pages_with_search_data_before_T,
           ROUND(AVG(CASE WHEN c.content_updated_date >= DATE '{T.date()}' THEN 1.0 ELSE 0 END), 3) AS share_updated_after_T,
           ROUND(AVG(CASE WHEN c.last_optimized_date  >= DATE '{T.date()}' THEN 1.0 ELSE 0 END), 3) AS share_optimized_after_T
    FROM (SELECT DISTINCT content_hash_id FROM daily
          WHERE report_date BETWEEN DATE '{FEAT_START.date()}' AND DATE '{FEAT_END.date()}'
            AND gsc_data_available IS TRUE) p
    JOIN dim_content c USING (content_hash_id)
''')

57 columns in the two tables; 12 touched or named — every one exists and has a bucket.
Everything else stays out by default.



,pages_with_search_data_before_T,share_updated_after_T,share_optimized_after_T
0,151185,0.826,0.249


**Answer 5, backed:** of the 151,185 pages with search data before T, **82.6%** already show a `content_updated_date` *after* T (24.9% for `last_optimized_date`). For most pages the column holds a date from that page's own future. Excluded.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Exactly **three checks**, all on the `month=2026-03` partition. Each states its claim, then proves it.

### Check 1/3 — the grain
**Claim:** one row = one `report_date × client_hash_id × content_hash_id`. If that holds, the row count
equals the number of distinct keys and no key appears twice.

In [4]:
check_grain = sql('''
    SELECT COUNT(*)                                                       AS rows_total,
           COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS distinct_keys,
           (SELECT COUNT(*) FROM (SELECT report_date, client_hash_id, content_hash_id
                                  FROM daily GROUP BY ALL HAVING COUNT(*) > 1))  AS keys_seen_twice
    FROM daily
''')
assert check_grain.at[0, 'rows_total'] == check_grain.at[0, 'distinct_keys'] and check_grain.at[0, 'keys_seen_twice'] == 0
check_grain

,rows_total,distinct_keys,keys_seen_twice
0,9841378,9841378,0


**Holds.** 9,841,378 rows = 9,841,378 distinct keys, and 0 keys appear twice. One row really is one page on one day.

### Check 2/3 — my slice: row count and date span
**Claim:** the partition covers 2026-03-01 → 2026-03-31. My slice — the rows inside my feature +
outcome windows — covers 2026-03-02 → 2026-03-29, 28 days.

In [5]:
check_span = sql(f'''
    SELECT 'whole month=2026-03 partition' AS slice, COUNT(*) AS rows,
           MIN(report_date) AS first_day, MAX(report_date) AS last_day, COUNT(DISTINCT report_date) AS days,
           COUNT(DISTINCT client_hash_id) AS clients, COUNT(DISTINCT content_hash_id) AS content_items
    FROM daily
    UNION ALL
    SELECT 'my slice: feature + outcome windows', COUNT(*),
           MIN(report_date), MAX(report_date), COUNT(DISTINCT report_date),
           COUNT(DISTINCT client_hash_id), COUNT(DISTINCT content_hash_id)
    FROM daily
    WHERE report_date BETWEEN DATE '{FEAT_START.date()}' AND DATE '{OUT_END.date()}'
''')
check_span

,slice,rows,first_day,last_day,days,clients,content_items
0,whole month=2026-03 partition,9841378,2026-03-01,2026-03-31,31,55,331437
1,my slice: feature + outcome windows,8902838,2026-03-02,2026-03-29,28,55,326193


**Holds.** The partition has 9,841,378 rows over 31 days (2026-03-01 → 2026-03-31), from 55 clients and 331,437 content items. My slice keeps **8,902,838 rows over exactly 28 days (2026-03-02 → 2026-03-29)**, still 55 clients and 326,193 items. The 5,244 extra items in the full month show up only on the unused days. Items enter the table from the day they are registered, so the feature frame keeps only pages tracked from the first day of the feature window.

### Check 3/3 — availability, filtered with `IS TRUE`
**Claim:** most rows carry no search data. `gsc_data_available` is TRUE only on rows where Search
Console reported the page that day, and `ga4_data_available` can be TRUE, FALSE **or NULL**.
`IS TRUE` keeps only rows that really have data: a NULL flag counts as "not available", never as data.
(`WHERE NOT flag` would quietly drop the NULL rows from the "not available" side too.)

In [6]:
check_avail = sql('''
    SELECT COUNT(*)                                             AS rows_total,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)  AS gsc_is_true,
           COUNT(*) FILTER (WHERE gsc_data_available IS FALSE) AS gsc_is_false,
           COUNT(*) FILTER (WHERE gsc_data_available IS NULL)  AS gsc_is_null,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)  AS ga4_is_true,
           COUNT(*) FILTER (WHERE ga4_data_available IS FALSE) AS ga4_is_false,
           COUNT(*) FILTER (WHERE ga4_data_available IS NULL)  AS ga4_is_null,
           COALESCE(SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS NOT TRUE), 0) AS impressions_on_rows_not_true
    FROM daily
''')
check_avail['gsc_survive_share'] = (check_avail['gsc_is_true'] / check_avail['rows_total']).round(3)
check_avail.T.rename(columns={0: 'rows'})

,rows
rows_total,9841378.000
gsc_is_true,3611061.000
gsc_is_false,6230317.000
gsc_is_null,0.000
ga4_is_true,413966.000
ga4_is_false,6408671.000
ga4_is_null,3018741.000
impressions_on_rows_not_true,0.000
gsc_survive_share,0.367


**Holds.** `gsc_data_available IS TRUE` keeps **3,611,061 of 9,841,378 rows (36.7%)**. The other 6,230,317 rows are FALSE and carry 0 impressions: "no search data that day", not "zero demand". The GSC flag has no NULLs. The GA4 flag does: 3,018,741 rows are NULL, 6,408,671 FALSE and only 413,966 TRUE. That is one more reason this slice stays GSC-only. Every feature below filters with `IS TRUE`.

### Five features (max) — each knowable at the decision moment

All five come from the feature window (before T), on rows where `gsc_data_available IS TRUE`.

| # | Feature | Built from | Knowable at T because… |
|---|---|---|---|
| 1 | `log_imp_pre` | log(1 + impressions, 03-02 → 03-15) | every day it sums ends before T |
| 2 | `pos_pre` | impression-weighted average position, skipping position 0 ("no data") | positions already reported for days before T |
| 3 | `active_days_pre` | days out of 14 with impressions | counted only over days before T |
| 4 | `momentum_pre` | log((impressions week 2 + 1) / (impressions week 1 + 1)), 03-09→03-15 vs 03-02→03-08 | it compares the past with the past; both weeks end before T |
| 5 | `age_days_at_T` | T − `content_created_date` | the page was created before T (asserted below), and a creation date does not move later |

The outcome window is aggregated in the same query but only becomes the **label**. It never enters
`FEATURES`.

In [7]:
frame = sql(f'''
    WITH pre AS (                 -- feature window: strictly before T
        SELECT client_hash_id, content_hash_id,
               MIN(report_date)                                                     AS first_seen,
               SUM(gsc_impressions)  FILTER (WHERE gsc_data_available IS TRUE)       AS imp_pre,
               SUM(gsc_impressions)  FILTER (WHERE gsc_data_available IS TRUE
                                             AND report_date <  DATE '2026-03-09')  AS imp_wk1,
               SUM(gsc_impressions)  FILTER (WHERE gsc_data_available IS TRUE
                                             AND report_date >= DATE '2026-03-09')  AS imp_wk2,
               SUM(gsc_sum_position) FILTER (WHERE gsc_data_available IS TRUE AND gsc_avg_position > 0)
                 / NULLIF(SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE
                                                       AND gsc_avg_position > 0), 0) AS pos_pre,
               COUNT(*)              FILTER (WHERE gsc_data_available IS TRUE)       AS active_days_pre
        FROM daily
        WHERE report_date BETWEEN DATE '{FEAT_START.date()}' AND DATE '{FEAT_END.date()}'
        GROUP BY ALL
    ),
    outcome AS (                  -- outcome window: label only
        SELECT client_hash_id, content_hash_id,
               COUNT(*)                                                         AS out_days_tracked,
               SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE)   AS imp_out
        FROM daily
        WHERE report_date BETWEEN DATE '{OUT_START.date()}' AND DATE '{OUT_END.date()}'
        GROUP BY ALL
    )
    SELECT p.*, COALESCE(o.out_days_tracked, 0) AS out_days_tracked, COALESCE(o.imp_out, 0) AS imp_out,
           c.content_created_date
    FROM pre p
    LEFT JOIN outcome o USING (client_hash_id, content_hash_id)
    LEFT JOIN dim_content c USING (content_hash_id)
    WHERE p.imp_pre >= 100
''')
n_volume = len(frame)
frame = frame[pd.to_datetime(frame['first_seen']) == FEAT_START]       # tracked for the whole feature window
n_full = len(frame)
frame = frame.dropna(subset=['pos_pre']).reset_index(drop=True)          # no valid position on any day

frame['log_imp_pre']   = np.log1p(frame['imp_pre'])
frame['momentum_pre']  = np.log((frame['imp_wk2'].fillna(0) + 1) / (frame['imp_wk1'].fillna(0) + 1))
frame['age_days_at_T'] = (T - pd.to_datetime(frame['content_created_date'])).dt.days
frame['declined_next_2w'] = (frame['imp_out'] < 0.8 * frame['imp_pre']).astype(int)

FEATURES = ['log_imp_pre', 'pos_pre', 'active_days_pre', 'momentum_pre', 'age_days_at_T']
LABEL = 'declined_next_2w'

assert (frame['age_days_at_T'] > 0).all(), 'a scored page was created after T'
assert (frame['out_days_tracked'] == 14).all(), 'a "decline" must never be a tracking gap'
assert frame[FEATURES].notna().all().all()

print(f'pages with >= 100 impressions before T: {n_volume:,}')
print(f'  ... tracked from the first day of the feature window: {n_full:,}')
print(f'  ... with a valid position: {len(frame):,} pages from {frame.client_hash_id.nunique()} clients')
print(f'label base rate (share that dropped > 20%): {frame[LABEL].mean():.3f}\n')
frame[FEATURES + [LABEL]].describe().round(2)

pages with >= 100 impressions before T: 76,259
  ... tracked from the first day of the feature window: 74,543
  ... with a valid position: 74,542 pages from 36 clients
label base rate (share that dropped > 20%): 0.337



,log_imp_pre,pos_pre,active_days_pre,momentum_pre,age_days_at_T,declined_next_2w
count,74542.00,74542.00,74542.00,74542.00,74542.00,74542.00
mean,6.48,12.60,13.52,0.02,186.84,0.34
std,1.23,12.96,1.26,0.76,121.30,0.47
min,4.62,0.02,1.00,-7.62,11.00,0.00
25%,5.48,4.31,14.00,-0.31,68.00,0.00
50%,6.35,7.39,14.00,-0.03,180.00,0.00
75%,7.32,16.28,14.00,0.27,251.00,1.00
max,11.97,119.43,14.00,8.21,479.00,1.00


### A quick honest score

The notebook-02 recipe: a simple model, a held-out test set, a base rate to compare against.
I split **by client** (`GroupShuffleSplit` on `client_hash_id`, seed 42), so test pages come from
sites the model never saw. Two numbers: ROC-AUC (quality of the whole ranking) and **precision@50**,
the ML-03 lane metric — of the 50 pages ranked riskiest, how many really dropped.

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42

def client_split(df, seed=SEED):
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=seed)
                  .split(df, groups=df['client_hash_id']))
    return df.iloc[tr], df.iloc[te]

def precision_at_k(y, scores, k=50):
    return float(np.asarray(y)[np.argsort(-np.asarray(scores))[:k]].mean())

def scored_test_set(df, cols, seed=SEED):
    train, test = client_split(df, seed)
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(train[cols], train[LABEL])
    return test.assign(score=model.predict_proba(test[cols])[:, 1])

def quick_score(df, cols, seed=SEED):
    test = scored_test_set(df, cols, seed)
    return {'n_features': len(cols), 'roc_auc': round(float(roc_auc_score(test[LABEL], test['score'])), 3),
            'precision@50': round(precision_at_k(test[LABEL], test['score']), 3),
            'test_base_rate': round(float(test[LABEL].mean()), 3),
            'test_pages': len(test), 'test_clients': int(test['client_hash_id'].nunique())}

honest = quick_score(frame, FEATURES)

# what sits at the top of the honest ranking?
ranked = scored_test_set(frame, FEATURES)
top50 = ranked.nlargest(50, 'score')
print(f"median momentum_pre: top 50 = {top50['momentum_pre'].median():.2f}   all test pages = {ranked['momentum_pre'].median():.2f}")
print(f"share of top 50 whose week-2 impressions were under half of week 1: {(top50['momentum_pre'] < np.log(0.5)).mean():.2f}\n")
pd.DataFrame([honest], index=['honest: 5 features'])

median momentum_pre: top 50 = -4.72   all test pages = -0.01
share of top 50 whose week-2 impressions were under half of week 1: 1.00



,n_features,roc_auc,precision@50,test_base_rate,test_pages,test_clients
honest: 5 features,5,0.711,0.96,0.361,36786,11


On 36,786 test pages from 11 clients the model never saw: **ROC-AUC 0.711, precision@50 0.96**, against a base rate of 0.361.

Read the 0.96 with care. Every one of the top 50 had already lost more than half of its impressions in week 2 (median `momentum_pre` −4.72, against −0.01 for all test pages). So precision@50 mostly says *a visible collapse tends to continue for two more weeks*. That is useful for a queue, but it is not a discovery. AUC 0.71 is the fairer summary of the whole ranking. It comes from a single split, and section 4 shows how much it moves. Directional, not final.

### The trap — one label-derived column, added on purpose

I add `imp_out_log` = log(1 + impressions in the outcome window). The label is computed from this
exact number, so it is information from **after** T. A model at T could never have it. Watch the score.

In [9]:
frame['imp_out_log'] = np.log1p(frame['imp_out'])          # LEAK: built from the outcome window

leaky = quick_score(frame, FEATURES + ['imp_out_log'])
pd.DataFrame([honest, leaky], index=['honest: 5 features', 'LEAK: 5 features + imp_out_log'])

,n_features,roc_auc,precision@50,test_base_rate,test_pages,test_clients
honest: 5 features,5,0.711,0.96,0.361,36786,11
LEAK: 5 features + imp_out_log,6,1.000,1.00,0.361,36786,11


**AUC jumps 0.711 → 1.000 and precision@50 0.96 → 1.00.** A perfect score, and none of it is skill. The label is `imp_out < 0.8 × imp_pre`, and with `imp_out_log` sitting next to `log_imp_pre` the model only has to learn that one line. At T the outcome-window impressions do not exist yet, so in real use this model would have nothing to run on. This is the notebook-02 lesson on real warehouse data: a score this good is a question to investigate, not a result.

**Now I delete it** and score again. The number I keep is the honest one.

In [10]:
frame = frame.drop(columns=['imp_out_log'])
assert 'imp_out_log' not in frame.columns
assert not set(FEATURES) & {'imp_out', 'out_days_tracked', LABEL}, 'no outcome-window column may be a feature'

final = quick_score(frame, FEATURES)
assert final == honest, 'after removing the leak the score must return to the honest number'
print('Honest number I keep ->', final)

Honest number I keep -> {'n_features': 5, 'roc_auc': 0.711, 'precision@50': 0.96, 'test_base_rate': 0.361, 'test_pages': 36786, 'test_clients': 11}


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation: a few clients dominate my slice.** The 74,542 scored pages come from only 36 clients. The largest client holds 23.2% of them, the top 3 hold 55.6% and the top 5 hold 73.9%. The decline rate itself ranges from 0.10 to 0.74 between clients (21 clients with ≥ 200 pages). So any pooled number is mostly a statement about a handful of sites.

The table below runs the same honest recipe on five different client splits. **ROC-AUC moves from 0.548 to 0.772** just from which 11 clients land in the test set. Before I trust a model in the modeling weeks, I need grouped cross-validation and per-client results, not one pooled score.

Also true, but smaller: two-week windows are short, so a 20% drop mixes real decline with noise and one-off events. Seasonality cannot be checked inside one month. And the slice is GSC-only.

In [11]:
share = frame['client_hash_id'].value_counts(normalize=True)
per_client = frame.groupby('client_hash_id')[LABEL].agg(pages='size', decline_rate='mean')
big = per_client[per_client['pages'] >= 200]

print(f'{len(share)} clients in the frame')
print(f'largest client holds {share.iloc[0]:.1%} of pages; top 3 hold {share.iloc[:3].sum():.1%}; top 5 hold {share.iloc[:5].sum():.1%}')
print(f'decline rate across the {len(big)} clients with >= 200 pages: '
      f'{big.decline_rate.min():.2f} to {big.decline_rate.max():.2f} (median {big.decline_rate.median():.2f})\n')

# the same honest recipe, five different client splits: how much does the number move?
swing = pd.DataFrame([quick_score(frame, FEATURES, seed=s) | {'seed': s} for s in [0, 1, 2, 3, 42]]).set_index('seed')
swing[['roc_auc', 'precision@50', 'test_base_rate', 'test_clients', 'test_pages']]

36 clients in the frame
largest client holds 23.2% of pages; top 3 hold 55.6%; top 5 hold 73.9%
decline rate across the 21 clients with >= 200 pages: 0.10 to 0.74 (median 0.37)



,roc_auc,precision@50,test_base_rate,test_clients,test_pages
seed,,,,,
0,0.772,1.00,0.317,11,21709
1,0.548,0.94,0.246,11,8314
2,0.574,0.90,0.359,11,18563
3,0.667,0.96,0.368,11,19419
42,0.711,0.96,0.361,11,36786


In [12]:
# Receipts: the numbers this notebook reports, written as JSON next to the other outputs.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'work' / 'notebooks').is_dir()), Path.cwd())
out_dir = ROOT / 'work' / 'outputs'
out_dir.mkdir(parents=True, exist_ok=True)

receipts = {
    'notebook': 'work/notebooks/w03_data_contract.ipynb',
    'release': 'FlyRank/internship-warehouse v20260703',
    'dev_month': DEV_MONTH, 'decision_moment_T': str(T.date()),
    'feature_window': [str(FEAT_START.date()), str(FEAT_END.date())],
    'outcome_window': [str(OUT_START.date()), str(OUT_END.date())],
    'check_1_grain': check_grain.iloc[0].to_dict(),
    'check_2_span': check_span.astype(str).to_dict(orient='records'),
    'check_3_availability': check_avail.iloc[0].to_dict(),
    'frame': {'pages': len(frame), 'clients': int(frame.client_hash_id.nunique()),
              'base_rate': round(float(frame[LABEL].mean()), 3), 'features': FEATURES},
    'score_honest': honest, 'score_with_leak': leaky, 'score_kept': final,
    'split_swing': swing[['roc_auc', 'precision@50']].reset_index().to_dict(orient='records'),
    'seed': SEED,
}
(out_dir / 'w03_data_contract_receipts.json').write_text(json.dumps(receipts, indent=2, default=lambda o: o.item() if hasattr(o, 'item') else str(o)))
print('wrote', (out_dir / 'w03_data_contract_receipts.json').relative_to(ROOT))

wrote work\outputs\w03_data_contract_receipts.json


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.